In [ ]:
from datetime import datetime
from datetime import timedelta
import logging, time, pymysql, os
import random
from contextlib import contextmanager

from faker import Faker

In [ ]:
def normalize_mysql_config(config: dict) -> dict:
    return {"host": config["host"], "port": int(config.get("port", 3306)), "user": config["user"], "password": config["password"], "connect_timeout": int(config.get("connect_timeout", 5)), "charset": config.get("charset", "utf8mb4")}


def connect_mysql(config: dict):
    c = normalize_mysql_config(config)
    return pymysql.connect(host=c["host"], port=c["port"], user=c["user"], password=c["password"], connect_timeout=c["connect_timeout"], charset=c["charset"], autocommit=True, cursorclass=pymysql.cursors.DictCursor)


@contextmanager
def mysql_conn(config: dict):
    conn = connect_mysql(config)
    try:
        yield conn
    finally:
        conn.close()


def exec_sql(config: dict, sql: str) -> None:
    with mysql_conn(config) as conn:
        with conn.cursor() as cur:
            for stmt in (s.strip() for s in sql.split(";")):
                if stmt: cur.execute(stmt)


def query_one(config: dict, sql: str):
    with mysql_conn(config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchone()


def wait_mysql(config: dict, retries: int = 60, sleep_sec: int = 2) -> None:
    h, p = config.get("host"), config.get("port", 3306);
    logging.info("Waiting MySQL %s:%s", h, p)
    last_err = None
    for _ in range(retries):
        try:
            if query_one(config, "SELECT 1") is not None: logging.info("MySQL is up %s:%s", h, p); return
        except Exception as e:
            last_err = e
        time.sleep(sleep_sec)
    logging.error("MySQL not ready %s:%s (last_error=%r)", h, p, last_err);
    raise SystemExit(1)


def mysql_major(config: dict) -> int:
    row = query_one(config, "SELECT VERSION() AS v");
    v = (row or {}).get("v", "0.0.0").split("-")[0];
    return int(v.split(".")[0]) if v and v[0].isdigit() else 0


def ensure_binlog_enabled(primary_: dict) -> None:
    row = query_one(primary_, "SHOW VARIABLES LIKE 'log_bin'");
    val = (row or {}).get("Value", "").upper()
    if val != "ON": raise RuntimeError("Primary log_bin is OFF. Enable binary logging for replication.")


def ensure_repl_user(primary_: dict, repl_user_: str, repl_pass_: str) -> None:
    logging.info("Ensure replication user on PRIMARY")
    sql = f"CREATE USER IF NOT EXISTS '{repl_user}'@'%' IDENTIFIED WITH mysql_native_password BY '{repl_pass_}';" \
          f"ALTER USER '{repl_user_}'@'%' IDENTIFIED WITH mysql_native_password BY '{repl_pass_}';" \
          f"GRANT REPLICATION SLAVE, REPLICATION CLIENT ON *.* TO '{repl_user_}'@'%';" \
          f"FLUSH PRIVILEGES;"
    exec_sql(primary, sql)


def get_master_status(primary_: dict) -> tuple[str, int]:
    logging.info("Fetch master status (SHOW MASTER STATUS)")
    row = query_one(primary_, "SHOW MASTER STATUS")
    if not row: raise RuntimeError("SHOW MASTER STATUS returned empty (binlog disabled or insufficient privileges).")
    file_, pos = row.get("File"), row.get("Position")
    if not file_ or pos is None: raise RuntimeError(f"Unexpected master status row: {row}")
    return str(file_), int(pos)


def stop_replica(replica_: dict) -> None:
    cmd = "STOP REPLICA" if mysql_major(replica_) >= 8 else "STOP SLAVE"
    try:
        exec_sql(replica_, cmd + ";")
    except Exception as e:
        logging.warning("Ignore stop error: %r", e)


def start_replica(replica_: dict) -> None:
    cmd = "START REPLICA" if mysql_major(replica_) >= 8 else "START SLAVE"
    exec_sql(replica_, cmd + ";")


def change_replication_source(replica_: dict, source_host_: str, source_port_: int, repl_user_: str, repl_pass_: str, binlog_file_: str, binlog_pos_: int) -> None:
    if mysql_major(replica_) >= 8:
        sql = f"CHANGE REPLICATION SOURCE TO SOURCE_HOST='{source_host_}',SOURCE_PORT={int(source_port_)},SOURCE_USER='{repl_user_}',SOURCE_PASSWORD='{repl_pass_}',SOURCE_LOG_FILE='{binlog_file_}',SOURCE_LOG_POS={int(binlog_pos_)};"
    else:
        sql = f"CHANGE MASTER TO MASTER_HOST='{source_host_}',MASTER_PORT={int(source_port_)},MASTER_USER='{repl_user_}',MASTER_PASSWORD='{repl_pass_}',MASTER_LOG_FILE='{binlog_file_}',MASTER_LOG_POS={int(binlog_pos_)};"
    exec_sql(replica_, sql)


def show_replica_status(replica_: dict) -> None:
    major = mysql_major(replica_)
    sql = "SHOW REPLICA STATUS" if major >= 8 else "SHOW SLAVE STATUS"
    row = query_one(replica_, sql)
    if not row: logging.warning("Replica status empty (not configured or no privileges)"); return
    keys = ["Replica_IO_Running", "Replica_SQL_Running", "Last_IO_Error", "Last_SQL_Error", "Source_Host", "Source_Log_File", "Read_Source_Log_Pos", "Relay_Source_Log_File", "Exec_Source_Log_Pos",
            "Slave_IO_Running", "Slave_SQL_Running", "Last_IO_Error", "Last_SQL_Error", "Master_Host", "Master_Log_File", "Read_Master_Log_Pos", "Relay_Master_Log_File", "Exec_Master_Log_Pos"]
    for k in keys:
        if k in row: logging.info("%s: %s", k, row.get(k))

## Mysql-Primary, Mysql-Replica 복제하기

In [ ]:
primary = {"host": "mysql-primary", "port": 3306, "user": "root", "password": "root", "database": "mmix"}
replica = {"host": "mysql-replication", "port": 3307, "user": "root", "password": "root"}

repl_user, repl_pass = "mmix", "mmix"
source_host, source_port = primary["host"], primary["port"]

wait_mysql(primary)
wait_mysql(replica)

ensure_binlog_enabled(primary)
ensure_repl_user(primary, repl_user, repl_pass)

binlog_file, binlog_pos = get_master_status(primary)
stop_replica(replica)
change_replication_source(replica, source_host, source_port, repl_user, repl_pass, binlog_file, binlog_pos)
start_replica(replica)
show_replica_status(replica)
logging.info("Done.")

## Mysql-Primary, Mysql-Replica 복제 확인

In [ ]:
def create_users(cursor, fake: Faker, count: int) -> list[int]:
    user_ids: list[int] = []
    for _ in range(count):
        cursor.execute("""
            INSERT INTO users (email, name, age, gender, job, address, signup) VALUES (%s, %s, %s, %s, %s,%s, %s)
            ON DUPLICATE KEY UPDATE name=VALUES(name), age=VALUES(age), gender=VALUES(gender), job=VALUES(job), address=VALUES(address), signup=VALUES(signup)""",
                       (fake.unique.email(), fake.name(), fake.random_int(min=10, max=100), random.choice(["M", "F"]), fake.job(), fake.address(), (datetime.now() - timedelta(days=random.randint(0, 365))).date().isoformat()))
        user_ids.append(cursor.lastrowid)
    return user_ids

In [ ]:
connection = pymysql.connect(user=primary["user"], password=primary["password"], host=primary["host"], database=primary["database"], autocommit=True, cursorclass=pymysql.cursors.DictCursor)
users = create_users(connection.cursor(), Faker("ko_KR"), 1)